<a href="https://colab.research.google.com/github/aiwithpavansama/Stages-of-learning-LLM-Data-pre-processing-from-Scratch-LLM-Base-/blob/main/Stage_1_Data_preprossing_from_Scratch(LLM_Base).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **#reading the words**

### Stage 1: Data Preparation

In [56]:
!pip install pypdf
from pypdf import PdfReader

In [57]:
# reading the text from the PDF and then getting first 100 characters

reader = PdfReader("/content/drive/MyDrive/The_Verdict.pdf")   # or "/path/to/file.pdf"

raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text() or ""

print("total number of characters",len(raw_text))
print(raw_text[:99])

total number of characters 21813
1
The Verdict
Edith Wharton
1908
Exported from Wikisource on May 20, 20242
I HAD always thought Jac


In [58]:
# splitting the words to create tokens

In [59]:
import re
result=re.split(r'(\s)',raw_text)
result = [item for item in result if item.strip()]
print(len(result))
print(result[:99])

3848
['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', 'May', '20,', '20242', 'I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius--', 'though', 'a', 'good', 'fellow', 'enough--so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that,', 'in', 'the', 'height', 'of', 'his', 'glory,', 'he', 'had', 'dropped', 'his', 'painting,', 'married', 'a', 'rich', 'widow,', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera.', '(Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence.)', '"The', 'height', 'of', 'his', 'glory"--that', 'was', 'what', 'the', 'women', 'called', 'it.', 'I', 'can', 'hear', 'Mrs.', 'Gideon', 'Thwing--his', 'last', 'Chicago', 'sitter--', 'deploring', 'his', 'unaccountable', 'abdication.', '"Of', 'course']


In [60]:
# we would have created a token on this one but we cannpt since we need camma and other special characters to be as a token so we further split the data so that orginal content is not missed

In [61]:
preprocessed=re.split(r'([,.:;?_!()\']|--|\s)',raw_text)
preprocessed=[item for item in preprocessed if item.strip()]
print(preprocessed[:30])

['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', 'May', '20', ',', '20242', 'I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough']


In [62]:
print(len(preprocessed))

4845


# **Step 2: Creating a Token ID's**:

In [63]:
# sorting in order and giving the token id's
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)
print("total number of unique words",vocab_size)

total number of unique words 1279


In [64]:
vocab={token:Integer for Integer,token in enumerate(all_words)}

In [65]:
for i, item in enumerate(vocab.items()):
  print(item)
  if i>=50:
    break

('!', 0)
('"', 1)
('"10', 2)
('"19', 3)
('"Ah', 4)
('"Be', 5)
('"Begin', 6)
('"By', 7)
('"Come', 8)
('"Destroyed', 9)
('"Don', 10)
('"Gisburns"', 11)
('"Grindles', 12)
('"Hang', 13)
('"Has', 14)
('"How', 15)
('"I', 16)
('"If', 17)
('"It', 18)
('"Jack', 19)
('"Money', 20)
('"Moon-dancers"', 21)
('"Mr', 22)
('"Mrs', 23)
('"My', 24)
('"Never', 25)
('"Of', 26)
('"Oh', 27)
('"Once', 28)
('"Only', 29)
('"Or', 30)
('"That', 31)
('"The', 32)
('"Then', 33)
('"There', 34)
('"This', 35)
('"We', 36)
('"Well', 37)
('"What', 38)
('"When', 39)
('"Why', 40)
('"Yes', 41)
('"You', 42)
('"but', 43)
('"deadening', 44)
('"dragged', 45)
('"effects"', 46)
('"interesting"', 47)
('"lift', 48)
('"obituary"', 49)
('"strongest', 50)


# **#implement Tokenizer class for encode and decode**

In [66]:
import re

# Define the upgraded tokenizer class
class SimpleTokenizer:
    def __init__(self, vocab=None):
        if vocab is not None:
            self.str_to_int = vocab
            self.int_to_str = {idx: token for token, idx in vocab.items()}
        else:
            self.str_to_int = {}
            self.int_to_str = {}

    def encode(self, text):
        if isinstance(text, str):
            # Tokenize on spaces, punctuation, and special tokens
            tokens = re.split(r'([,.:;?_!()\']|--|\s|<\|endoftext\|>|<\|unk\|>)', text)
            tokens = [item for item in tokens if item and item.strip()]
        else:
            tokens = text

        encoded_ids = []
        for token in tokens:
            if token in self.str_to_int:
                encoded_ids.append(self.str_to_int[token])
            elif '<|unk|>' in self.str_to_int:
                encoded_ids.append(self.str_to_int['<|unk|>'])
            else:
                # If <|unk|> is not configured in vocab, skip the unknown token
                pass
        return encoded_ids

    def decode(self, token_ids):
        tokens = [self.int_to_str[idx] for idx in token_ids if idx in self.int_to_str]
        text = " ".join(tokens)
        # Clean up spaces before punctuation marks
        text = re.sub(r"\s+([,.:;?(!'\"])", r"\1", text)
        return text

# Fix: Create vocab_with_specials before using it
all_words_with_specials = sorted(list(set(preprocessed)))
all_words_with_specials.extend(["<|unk|>", "<|endoftext|>"])
vocab_with_specials = {token: integer for integer, token in enumerate(all_words_with_specials)}

# Now initialize the upgraded tokenizer successfully
tokenizer = SimpleTokenizer(vocab=vocab_with_specials)

# Test with 'hello' and '<|endoftext|>'
test_text = "hello It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>"
ids = tokenizer.encode(test_text)
print("Encoded IDs (first ID is <|unk|>):", ids)
print("Decoded output:", tokenizer.decode(ids))

Encoded IDs (first ID is <|unk|>): [1279, 127, 1223, 835, 1150, 1144, 1272, 720, 1125, 56, 683, 1127, 402, 846, 201, 549, 1233, 53, 680, 852, 1127, 157, 56, 701, 1087, 844, 1157, 783, 1157, 1262, 1248, 113, 629, 605, 1193, 664, 876, 58, 147, 963, 56, 701, 956, 1223, 201, 1117, 927, 1280]
Decoded output: <|unk|> It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>


### Upgraded Tokenizer with Special Tokens Support

We will now recreate our vocabulary list and include two special tokens:
* `<|unk|>`: Used to represent any word that was not present in our training corpus (like `'hello'`).
* `<|endoftext|>`: Used to mark boundaries between different documents or sections.

In [67]:
# 1. Create a new sorted list of unique words and append the special tokens
all_words_with_specials = sorted(list(set(preprocessed)))
all_words_with_specials.extend(["<|unk|>", "<|endoftext|>"])

# 2. Build the updated vocabulary mapping
vocab_with_specials = {token: integer for integer, token in enumerate(all_words_with_specials)}

print("New vocab size:", len(vocab_with_specials))
print("ID of <|unk|>:", vocab_with_specials["<|unk|>"])
print("ID of <|endoftext|>:", vocab_with_specials["<|endoftext|>"])

New vocab size: 1281
ID of <|unk|>: 1279
ID of <|endoftext|>: 1280


In [68]:
import re

class SimpleTokenizer:
    def __init__(self, vocab=None):
        if vocab is not None:
            self.str_to_int = vocab
            self.int_to_str = {idx: token for token, idx in vocab.items()}
        else:
            self.str_to_int = {}
            self.int_to_str = {}

    def encode(self, text):
        if isinstance(text, str):
            # Tokenize on spaces, punctuation, and special tokens
            tokens = re.split(r'([,.:;?_!()\']|--|\s|<\|endoftext\|>|<\|unk\|>)', text)
            tokens = [item for item in tokens if item and item.strip()]
        else:
            tokens = text

        encoded_ids = []
        for token in tokens:
            if token in self.str_to_int:
                encoded_ids.append(self.str_to_int[token])
            elif '<|unk|>' in self.str_to_int:
                encoded_ids.append(self.str_to_int['<|unk|>'])
            else:
                # If <|unk|> is not configured in vocab, skip the unknown token
                pass
        return encoded_ids

    def decode(self, token_ids):
        tokens = [self.int_to_str[idx] for idx in token_ids if idx in self.int_to_str]
        text = " ".join(tokens)
        # Clean up spaces before punctuation marks
        text = re.sub(r"\s+([,.:;?(!'\"])", r"\1", text)
        return text

# Initialize the upgraded tokenizer using the vocabulary with special tokens
tokenizer = SimpleTokenizer(vocab=vocab_with_specials)

In [69]:
# Test with 'hello' and '<|endoftext|>'
test_text = "hello It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>"
ids = tokenizer.encode(test_text)
print("Encoded IDs (first ID is <|unk|>):", ids)
print("Decoded output:", tokenizer.decode(ids))

Encoded IDs (first ID is <|unk|>): [1279, 127, 1223, 835, 1150, 1144, 1272, 720, 1125, 56, 683, 1127, 402, 846, 201, 549, 1233, 53, 680, 852, 1127, 157, 56, 701, 1087, 844, 1157, 783, 1157, 1262, 1248, 113, 629, 605, 1193, 664, 876, 58, 147, 963, 56, 701, 956, 1223, 201, 1117, 927, 1280]
Decoded output: <|unk|> It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem <|endoftext|>


In [70]:
# 1. Create a new sorted list of unique words and append the special tokens
all_words_with_specials = sorted(list(set(preprocessed)))
all_words_with_specials.extend(["<|unk|>", "<|endoftext|>"])

# 2. Build the updated vocabulary mapping
vocab_with_specials = {token: integer for integer, token in enumerate(all_words_with_specials)}

print("New vocab size:", len(vocab_with_specials))
print("ID of <|unk|>:", vocab_with_specials["<|unk|>"])
print("ID of <|endoftext|>:", vocab_with_specials["<|endoftext|>"])

New vocab size: 1281
ID of <|unk|>: 1279
ID of <|endoftext|>: 1280


In [71]:
tokenizer=SimpleTokenizer(vocab)

In [72]:
text =""" Pavan Kalyan sama It was not till three years later that, in the course of a few
weeks' idling on the Riviera, it suddenly occurred to me to
wonder why Gisburn had given up his painting. On
reflection, it really was a tempting problem """

In [73]:
ids=tokenizer.encode(text)
print(ids)

[127, 1223, 835, 1150, 1144, 1272, 720, 1125, 56, 683, 1127, 402, 846, 201, 549, 1233, 53, 680, 852, 1127, 157, 56, 701, 1087, 844, 1157, 783, 1157, 1262, 1248, 113, 629, 605, 1193, 664, 876, 58, 147, 963, 56, 701, 956, 1223, 201, 1117, 927]


In [74]:
tokenizer.decode(ids)

"It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem"

### Explanation: Why "Pavan Kalyan" was skipped

The reason no IDs were generated for `Pavan` and `Kalyan` is because the `tokenizer` in cell `3jrFpvtghisD` was initialized with the original `vocab` which doesn't contain `<|unk|>` or those specific names.

If we re-initialize the tokenizer using `vocab_with_specials` (which contains `<|unk|>`), both words will map to the `<|unk|>` token ID (`1279`):

In [75]:
# Re-initialize tokenizer with the vocabulary that contains special tokens
tokenizer_with_specials = SimpleTokenizer(vocab=vocab_with_specials)

# Encode the text containing unseen words
ids_demo = tokenizer_with_specials.encode(text)
print("Encoded IDs (Notice the first two IDs are 1279 for '<|unk|>'):", ids_demo[:5])
print("Decoded text:", tokenizer_with_specials.decode(ids_demo))

Encoded IDs (Notice the first two IDs are 1279 for '<|unk|>'): [1279, 1279, 1279, 127, 1223]
Decoded text: <|unk|> <|unk|> <|unk|> It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem


### 1. Word-Based Tokenizer
* **Meaning**: Splits text into tokens based on individual words. This is typically done using whitespace and punctuation marks as boundaries.
* **Pros**: Highly intuitive; tokens map directly to meaningful concepts/words.
* **Cons**: Leads to a massive vocabulary size. It cannot handle out-of-vocabulary (OOV) words (any unseen word, like spelling mistakes or new words, maps to `<|unk|>`).

### 2. Character-Based Tokenizer
* **Meaning**: Splits text into individual characters (letters, numbers, and symbols).
* **Pros**: Tiny vocabulary size (usually just the alphabet and punctuation) and zero Out-Of-Vocabulary (OOV) words since any word can be spelled out.
* **Cons**: The sequence length becomes extremely long (each word contains multiple tokens), making it harder for models to learn long-range relationships between words.

### 3. Byte Pair Encoding (BPE) Tokenizer
* **Meaning**: A subword-level tokenizer. It starts at the character level and iteratively merges the most frequently co-occurring pairs of characters/subwords into new tokens until a target vocabulary size is met.
* **Pros**: Strikes a perfect balance. It keeps common words whole (e.g., `"the"`), but breaks unknown or rare words down into meaningful subword units (e.g., `"unbelievable"` -> `["un", "believ", "able"]`). It completely avoids the OOV problem without inflating sequence lengths.

### Why We Use `tiktoken`

While we can build simple character or word-based tokenizers from scratch (like our `SimpleTokenizer`), modern Large Language Models (LLMs) like GPT-4, GPT-3.5, and others use advanced subword tokenization algorithms. **`tiktoken`** is a fast Byte Pair Encoding (BPE) tokenizer developed by OpenAI.

Here is why we use it:

1. **High Performance & Speed**: `tiktoken` is written in Rust, making it extremely fast compared to pure Python tokenizers. It can handle massive text corpora very quickly.
2. **No Out-Of-Vocabulary (OOV) Errors**: Instead of dropping or replacing unknown words (like "Pavan Kalyan" mapping to `<|unk|>`), it breaks unknown or rare words down into smaller, recognizable subword units or bytes.
3. **Exact Alignment with OpenAI Models**: If you are working with GPT models, `tiktoken` guarantees that the tokens and IDs match the exact token counts and representations used by the model APIs. This is crucial for precise cost estimation and managing context limits.

In [76]:
!pip3 install tiktoken

In [77]:
import importlib
import tiktoken

In [78]:
tokenizer=tiktoken.get_encoding("gpt2")

**The line tokenizer = tiktoken.get_encoding("gpt2")**
 initializes the Byte Pair Encoding (BPE) tokenizer configuration specifically trained for OpenAI's GPT-2 model family. Here is exactly what this line does:

Loads the GPT-2 Vocabulary: It fetches the pre-trained dictionary of 50,257 unique tokens that GPT-2 was trained on.
Retrieves BPE Merging Rules: It configures the rules used to group individual characters or bytes together into high-frequency subword patterns.
Instantiates an Encoder Object: It returns a Python object equipped with optimized methods like .encode() (to turn text into a list of integers) and .decode() (to reconstruct the original text from token integers)

In [79]:
text_to_encode=vocab_with_specials

In [80]:
# Fix the error by explicitly setting text_to_encode to our target text string first
text_to_encode = "peter parker is one of the best super hero  in the marvel series"

gpt_ids = tokenizer.encode(text_to_encode)
print("Token IDs:", gpt_ids)

Token IDs: [79, 2357, 3952, 263, 318, 530, 286, 262, 1266, 2208, 4293, 220, 287, 262, 22672, 2168]


In [81]:
gpt_decode_text=tokenizer.decode(gpt_ids)
print(gpt_decode_text)

peter parker is one of the best super hero  in the marvel series


### Visualizing Byte Pair Encoding (BPE) for `text_to_encode`

To see BPE in action, we can take each token ID generated by `tiktoken` and decode it individually. This shows us exactly where the tokenizer splits words (including spaces, which are treated as part of the tokens):

In [82]:
# We use the GPT-2 tokenizer we initialized earlier
gpt_ids = tokenizer.encode(text_to_encode)

print(f"Original Text: \"{text_to_encode}\"\n")
print("--- BPE Token Breakdown ---")
print(f"{'Token ID':<10} | {'Token (Decoded String)':<25} | {'Notes'}")
print("-" * 60)

for token_id in gpt_ids:
    decoded_piece = tokenizer.decode([token_id])
    # Help visualize spaces by replacing them with a visible character
    visible_piece = decoded_piece.replace(" ", "•")

    note = "Whole word" if decoded_piece.strip() == decoded_piece else "Subword / Includes Space"
    print(f"{token_id:<10} | {repr(visible_piece):<25} | {note}")

Original Text: "peter parker is one of the best super hero  in the marvel series"

--- BPE Token Breakdown ---
Token ID   | Token (Decoded String)    | Notes
------------------------------------------------------------
79         | 'p'                       | Whole word
2357       | 'eter'                    | Whole word
3952       | '•park'                   | Subword / Includes Space
263        | 'er'                      | Whole word
318        | '•is'                     | Subword / Includes Space
530        | '•one'                    | Subword / Includes Space
286        | '•of'                     | Subword / Includes Space
262        | '•the'                    | Subword / Includes Space
1266       | '•best'                   | Subword / Includes Space
2208       | '•super'                  | Subword / Includes Space
4293       | '•hero'                   | Subword / Includes Space
220        | '•'                       | Subword / Includes Space
287        | '•in'             

# Creating input traget Pairs

In [83]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    # tokenize the entire text
    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    # use a sliding window to chunk the book into overlapping sequences of max_length
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    # Return the pre-constructed tensors directly to avoid redundant tensor creation warnings
    return self.input_ids[idx], self.target_ids[idx]

### How the Sliding Window Creates Input-Output Pairs

To train an LLM to predict the next token in a sequence, we use a **sliding window** approach. This technique splits a continuous stream of token IDs into multiple training examples consisting of an input sequence ($X$) and its corresponding shifted target sequence ($Y$).

#### Key Parameters:
* **`max_length` (Context Window)**: The number of tokens in each training sample's input chunk.
* **`stride`**: The step size of our sliding window. A stride of `1` means the window shifts forward by exactly one token to create the next example.

#### Visual Demonstration:

Imagine we have a tokenized sequence: `[50, 51, 52, 53, 54, 55, 56]` and set `max_length = 4`, `stride = 1`:

* **Step 1 (Index 0)**:
  * **Input (X)**: `[50, 51, 52, 53]`
  * **Target (Y)**: `[51, 52, 53, 54]` (shifted right by 1)

* **Step 2 (Index 1 - shifted by 1)**:
  * **Input (X)**: `[51, 52, 53, 54]`
  * **Target (Y)**: `[52, 53, 54, 55]`

* **Step 3 (Index 2 - shifted by another 1)**:
  * **Input (X)**: `[52, 53, 54, 55]`
  * **Target (Y)**: `[53, 54, 55, 56]`

This overlapping sequence generation allows the model to learn what token comes next from multiple positions within the same context chunk, maximizing the efficiency of our training data.

In [84]:
# 1. Convert the vocabulary keys from vocab_with_specials dictionary back into a single continuous text string
vocab_text = " ".join(list(vocab_with_specials.keys()))

# 2. Instantiate the TextDataset using the vocabulary string with a sliding window of max_length=4 and stride=1
vocab_dataset = TextDataset(vocab_text, tokenizer, max_length=4, stride=1)

print(f"Total training examples generated from vocab_with_specials: {len(vocab_dataset)}\n")

# 3. Print the first 5 input-target pairs to verify
for i in range(min(5, len(vocab_dataset))):
    x, y = vocab_dataset[i]
    print(f"--- Batch / Pair {i+1} ---")
    print(f"Input (X) IDs:   {x.tolist()}")
    print(f"Target (Y) IDs:  {y.tolist()}")
    print(f"Input tokens:    {[tokenizer.decode([token]) for token in x.tolist()]}")
    print(f"Target tokens:   {[tokenizer.decode([token]) for token in y.tolist()]}")
    print()

Total training examples generated from vocab_with_specials: 1753

--- Batch / Pair 1 ---
Input (X) IDs:   [0, 366, 366, 940]
Target (Y) IDs:  [366, 366, 940, 366]
Input tokens:    ['!', ' "', ' "', '10']
Target tokens:   [' "', ' "', '10', ' "']

--- Batch / Pair 2 ---
Input (X) IDs:   [366, 366, 940, 366]
Target (Y) IDs:  [366, 940, 366, 1129]
Input tokens:    [' "', ' "', '10', ' "']
Target tokens:   [' "', '10', ' "', '19']

--- Batch / Pair 3 ---
Input (X) IDs:   [366, 940, 366, 1129]
Target (Y) IDs:  [940, 366, 1129, 366]
Input tokens:    [' "', '10', ' "', '19']
Target tokens:   ['10', ' "', '19', ' "']

--- Batch / Pair 4 ---
Input (X) IDs:   [940, 366, 1129, 366]
Target (Y) IDs:  [366, 1129, 366, 10910]
Input tokens:    ['10', ' "', '19', ' "']
Target tokens:   [' "', '19', ' "', 'Ah']

--- Batch / Pair 5 ---
Input (X) IDs:   [366, 1129, 366, 10910]
Target (Y) IDs:  [1129, 366, 10910, 366]
Input tokens:    [' "', '19', ' "', 'Ah']
Target tokens:   ['19', ' "', 'Ah', ' "']



In [85]:
from torch.utils.data import DataLoader

# 1. Initialize the PyTorch DataLoader with batch size 4 and shuffling enabled
batch_size = 4
data_loader = DataLoader(
    vocab_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

# 2. Extract and inspect the first batch
data_iter = iter(data_loader)
first_batch_x, first_batch_y = next(data_iter)

print(f"DataLoader established!")
print(f"Batch Input (X) shape:  {first_batch_x.shape}")
print(f"Batch Target (Y) shape: {first_batch_y.shape}\n")

print("--- Decoded Batch Samples ---")
for i in range(batch_size):
    x_ids = first_batch_x[i].tolist()
    y_ids = first_batch_y[i].tolist()
    print(f"Sample {i+1} in Batch:")
    print(f"  Input text:  {repr(tokenizer.decode(x_ids))}")
    print(f"  Target text: {repr(tokenizer.decode(y_ids))}")
    print()

DataLoader established!
Batch Input (X) shape:  torch.Size([4, 4])
Batch Target (Y) shape: torch.Size([4, 4])

--- Decoded Batch Samples ---
Sample 1 in Batch:
  Input text:  ' hostess hot-'
  Target text: 'ess hot-house'

Sample 2 in Batch:
  Input text:  'dragged "effects'
  Target text: 'agged "effects"'

Sample 3 in Batch:
  Input text:  ' A Abigor About'
  Target text: ' Abigor About Adam'

Sample 4 in Batch:
  Input text:  'isource Yes You'
  Target text: 'ource Yes You Zh'



### What is a Token Embedding?

Once our tokenizer converts raw text into numerical **Token IDs** (integers like `[79, 2357, 3952]`), these integers cannot be used directly by deep learning models for mathematical calculations.

**Token Embedding** is the next step in the pipeline that converts these abstract integer IDs into dense, continuous vectors of real numbers (floats).

#### How it works:

1. **Embedding Matrix**: The model maintains a lookup table (matrix) of size `(vocab_size, embedding_dimension)`. For example, with GPT-2, the vocabulary size is `50,257` and the embedding dimension might be `768` floats per token.
2. **ID as an Index**: When we pass a Token ID (e.g., `79`), the model acts like it is looking up row index `79` in this massive matrix.
3. **Semantic Representation**: Initially, these vectors are randomized. During training, the model adjusts the floating-point values so that tokens with similar meanings, grammatical roles, or contexts end up with vectors that are geometrically close to each other in high-dimensional space (e.g., `'king'` - `'man'` + `'woman'` ≈ `'queen'`).

In [86]:
import torch
import torch.nn as nn

# 1. Define the vocabulary size and embedding dimension
# We use the standard GPT-2 vocabulary size (50,257) which aligns with our tiktoken tokenizer
vocab_size = 50257
embedding_dim = 256  # Map each token ID to a 256-dimensional continuous vector

In [87]:
# 2. Instantiate the PyTorch Embedding Layer
token_embedding_layer = nn.Embedding(vocab_size, embedding_dim)
print(token_embedding_layer)

Embedding(50257, 256)


In [88]:
# 3. Retrieve a single batch from our established DataLoader
data_iter = iter(data_loader)
batch_x, batch_y = next(data_iter)

In [89]:
# 4. Pass the input token IDs through the embedding layer
batch_embeddings_x = token_embedding_layer(batch_x)
batch_embeddings_y = token_embedding_layer(batch_y)

# 5. Inspect the output shapes
print(f"Original Token ID batch shape:       {batch_x.shape} (batch_size, sequence_length)")
print(f"Resulting Input Embeddings shape:    {batch_embeddings_x.shape} (batch_size, sequence_length, embedding_dim)")
print(f"Resulting Target Embeddings shape:   {batch_embeddings_y.shape} (batch_size, sequence_length, embedding_dim)")

Original Token ID batch shape:       torch.Size([4, 4]) (batch_size, sequence_length)
Resulting Input Embeddings shape:    torch.Size([4, 4, 256]) (batch_size, sequence_length, embedding_dim)
Resulting Target Embeddings shape:   torch.Size([4, 4, 256]) (batch_size, sequence_length, embedding_dim)


# Adding Positional Encoding to our Embeddings**
Why We Use Positional Encoding
Although Token Embeddings successfully map words (like "weekly", "weeks") to coordinates in a continuous semantic space, they suffer from one fatal flaw in standard Transformer architectures:

Transformers are permutation-invariant: Because Self-Attention calculates relationships between tokens in parallel (unlike sequential models like RNNs or LSTMs), the model sees a sequence of tokens as an unordered bag of words.
Order changes meaning: The sentences "The dog chased the cat" and "The cat chased the dog" contain the exact same token IDs. Without positional information, a Transformer would compute identical mathematical representations for both sentences.
Positional Encoding fixes this by adding a unique vector to each token's embedding based on its index (position) in the sequence. This informs the model where each token resides, allowing it to distinguish order and structure.

Let's implement the standard GPT-style Trainable Absolute Positional Embedding layer and add it directly to our token embeddings.

Positional Encoding Added Successfully!
The code executed correctly. Here is exactly what we accomplished in this step:

Position IDs: We created a simple tracking tensor tensor([0, 1, 2, 3]) representing the index of each word inside our context window.
Positional Embeddings: The model mapped each of those four positions to a unique, continuous 256-dimensional vector.
Summation: By adding the token embeddings (torch.Size([4, 4, 256])) and positional embeddings (torch.Size([4, 256])), PyTorch automatically broadcasted the values to yield a final combined embedding tensor of shape torch.Size([4, 4, 256]).

In [90]:
import torch
import torch.nn as nn

# 1. Define configuration
max_length = 4  # Context window size from earlier
max_seq_len = max_length
pos_vocab_size = max_seq_len

# 2. Instantiate the Trainable Positional Embedding layer
# (Each position 0, 1, 2, 3 gets its own 256-dimensional vector)
position_embedding_layer = nn.Embedding(pos_vocab_size, embedding_dim)

# 3. Create position IDs for our batch
# We need a tensor representing [0, 1, 2, 3] for each sequence in our batch of size 4
positions = torch.arange(max_seq_len, dtype=torch.long)
print(f"Position IDs: {positions}")

# 4. Extract positional embeddings
pos_embeddings = position_embedding_layer(positions)  # Shape: (sequence_length, embedding_dim)
print(f"Positional Embeddings shape: {pos_embeddings.shape}\n")

# 5. Add Token Embeddings + Positional Embeddings
# PyTorch automatically broadcasts pos_embeddings (shape [4, 256]) to match batch_embeddings_x (shape [4, 4, 256])
final_embeddings_x = batch_embeddings_x + pos_embeddings

print(f"Input Token Embeddings shape:  {batch_embeddings_x.shape}")
print(f"Final combined Embeddings shape: {final_embeddings_x.shape}")

Position IDs: tensor([0, 1, 2, 3])
Positional Embeddings shape: torch.Size([4, 256])

Input Token Embeddings shape:  torch.Size([4, 4, 256])
Final combined Embeddings shape: torch.Size([4, 4, 256])


### Stage 2: Attention Mechanism

Now that we have prepared our input tokens, generated embeddings, and added positional representations, the next critical phase in building our LLM from scratch is the **Attention Mechanism**.

#### What is the Attention Mechanism?
In a standard neural network, inputs are processed independently. However, in language, the meaning of a word depends heavily on its context (the other words around it). The Attention Mechanism allows the model to dynamically focus on different parts of the input sequence when processing a specific token.

* **Self-Attention**: Allows each token in a sequence to interact with and "pay attention" to every other token, calculating contextualized representations.
* **Scaled Dot-Product Attention**: The mathematical formulation that calculates relevance scores between tokens using **Queries (Q)**, **Keys (K)**, and **Values (V)**.
* **Causal/Masked Attention**: Restricts tokens from looking ahead at future tokens, ensuring the model only uses past and current context during next-token prediction.

* **Multi-head Attention**:is a core component of the Transformer neural network architecture introduced in the 2017 paper "Attention Is All You Need". It extends basic self-attention by running multiple attention mechanisms (called "heads") in paralle

### Attention Mechanism Concepts

*   Simplified attention mechanism:



# self attention with trainable weights: trainable weight metrics are crucial for good context vectors.


The code has executed successfully! Here is a breakdown of what we just achieved:

Input Vector ($X$$X$): We started with our final combined embeddings of shape torch.Size([4, 4, 256]) (representing 4 batches, a sequence length of 4 tokens, and 256 embedding dimensions).
Trainable Projections: We defined three separate linear layers (W_query, W_key, and W_value) with output dimension 16 and no bias terms.

**Computation of Q, K, V:**
Query (Q): What a token is looking for.
Key (K): What information a token contains (used for matching with Queries).
Value (V): The actual semantic content that gets aggregated based on the match between Q and K.
All three result tensors successfully match the target shape torch.Size([4, 4, 16]).

In [91]:
import torch
import torch.nn as nn

# Let's use our final combined embeddings from the previous step
# final_embeddings_x has shape: (batch_size, sequence_length, embedding_dim) -> (4, 4, 256)
x = final_embeddings_x
batch_size, seq_len, d_in = x.shape

# Define the target dimension for Q, K, V projection
d_out = 16  # projecting down to a smaller dimension for demonstration

# 1. Define trainable projection layers
W_query = nn.Linear(d_in, d_out, bias=False)
W_key = nn.Linear(d_in, d_out, bias=False)
W_value = nn.Linear(d_in, d_out, bias=False)

# 2. Project inputs to compute Queries, Keys, and Values
queries = W_query(x)  # Shape: (batch_size, seq_len, d_out)
keys = W_key(x)      # Shape: (batch_size, seq_len, d_out)
values = W_value(x)  # Shape: (batch_size, seq_len, d_out)

print(f"Input shape (X): {x.shape} (batch_size, seq_len, d_in)")
print(f"Query shape (Q): {queries.shape} (batch_size, seq_len, d_out)")
print(f"Key shape (K):   {keys.shape} (batch_size, seq_len, d_out)")
print(f"Value shape (V): {values.shape} (batch_size, seq_len, d_out)")

Input shape (X): torch.Size([4, 4, 256]) (batch_size, seq_len, d_in)
Query shape (Q): torch.Size([4, 4, 16]) (batch_size, seq_len, d_out)
Key shape (K):   torch.Size([4, 4, 16]) (batch_size, seq_len, d_out)
Value shape (V): torch.Size([4, 4, 16]) (batch_size, seq_len, d_out)


In [92]:
import math
import torch.nn.functional as F

# 1. Compute raw similarity scores using matrix multiplication
# queries: (batch_size, seq_len, d_out) -> (4, 4, 16)
# keys: (batch_size, seq_len, d_out) -> (4, 4, 16)
# We transpose the keys tensor along the sequence length and projection dimension to compute dot products
raw_scores = torch.bmm(queries, keys.transpose(1, 2))

# 2. Scale scores by the square root of the key dimension (d_out = 16, so sqrt(d_out) = 4)
d_k = keys.shape[-1]
scaled_scores = raw_scores / math.sqrt(d_k)

# 3. Apply softmax to get normalized attention weights along the last dimension (summing to 1 for each query)
attention_weights = F.softmax(scaled_scores, dim=-1)

print(f"Raw Scores shape:         {raw_scores.shape} (batch_size, seq_len, seq_len)")
print(f"Attention Weights shape:  {attention_weights.shape} (batch_size, seq_len, seq_len)")
print("\nAttention Weights for the first sequence in the batch (each row sums to 1.0):")
display(attention_weights[0])

Raw Scores shape:         torch.Size([4, 4, 4]) (batch_size, seq_len, seq_len)
Attention Weights shape:  torch.Size([4, 4, 4]) (batch_size, seq_len, seq_len)

Attention Weights for the first sequence in the batch (each row sums to 1.0):


tensor([[0.1912, 0.2734, 0.3183, 0.2170],
        [0.1312, 0.4880, 0.2089, 0.1719],
        [0.2023, 0.1605, 0.3405, 0.2968],
        [0.2182, 0.3377, 0.0930, 0.3510]], grad_fn=<SelectBackward0>)

### Why We Scale the Dot Product by $\sqrt{d_k}$

In the self-attention formula, the dot product of the Query ($Q$) and Key ($K$) is divided by the scaling factor $\sqrt{d_k}$:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right)V$$

Here is why this scaling factor is mathematically necessary:

#### 1. Preventing Large Dot Products
Assume that the components of our Query vector $q$ and Key vector $k$ are independent random variables with a mean of $0$ and variance of $1$:
$$\mathbb{E}[q_i] = 0, \quad \text{Var}(q_i) = 1$$
$$\mathbb{E}[k_i] = 0, \quad \text{Var}(k_i) = 1$$

Their dot product is the sum of $d_k$ individual products:
$$q \cdot k = \sum_{i=1}^{d_k} q_i k_i$$

By mathematical expectation and variance rules, the variance of this sum grows linearly with the dimension of the vectors ($d_k$):
$$\text{Var}(q \cdot k) = d_k$$

As the dimensionality $d_k$ becomes very large, the variance of the raw dot products becomes extremely large, causing some dot products to become extremely positive or extremely negative.

#### 2. Avoiding the Softmax Vanishing Gradient Problem
When the input values to the **Softmax** function are very large in magnitude, the softmax function behaves like a "hardmax" (assigning a probability of almost $1.0$ to the maximum value and $0.0$ to all others).

When this happens, the gradients of the softmax function with respect to its inputs become **extremely close to zero** (vanishing gradients). During training, this halts backpropagation through the attention weights, preventing the model from learning.

#### 3. How Scaling Solves This
By dividing the dot product by $\sqrt{d_k}$, we scale the variance of the dot product back down to $1$:
$$\text{Var}\left(\frac{q \cdot k}{\sqrt{d_k}}\right) = \frac{1}{d_k} \text{Var}(q \cdot k) = \frac{d_k}{d_k} = 1$$

This keeps the inputs to the softmax function in a reasonable range, ensuring stable gradients and smooth weight updates during training.

In [93]:
 import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleSelfAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        # Trainable weight projection matrices
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_in)
        keys = self.W_key(x)          # (B, T, d_out)
        queries = self.W_query(x)      # (B, T, d_out)
        values = self.W_value(x)        # (B, T, d_out)

        # 1. Compute raw similarity scores
        # bmm computes batch matrix multiplication: (B, T, d_out) x (B, d_out, T) -> (B, T, T)
        raw_scores = torch.bmm(queries, keys.transpose(1, 2))

        # 2. Scale scores by the square root of key dimension
        d_k = keys.shape[-1]
        scaled_scores = raw_scores / math.sqrt(d_k)

        # 3. Softmax to obtain attention weights
        attention_weights = F.softmax(scaled_scores, dim=-1)

        # 4. Compute final context vectors (B, T, T) x (B, T, d_out) -> (B, T, d_out)
        context_vectors = torch.bmm(attention_weights, values)

        return context_vectors

# --- Demo of the attention layer in action ---
# Initialize our self-attention module
attention_layer = SimpleSelfAttention(d_in=256, d_out=16)

# Pass our combined embeddings through the module
output_context_vectors = attention_layer(final_embeddings_x)

print("Input Embeddings Shape:", final_embeddings_x.shape)
print("Output Context Vectors Shape:", output_context_vectors.shape)

Input Embeddings Shape: torch.Size([4, 4, 256])
Output Context Vectors Shape: torch.Size([4, 4, 16])


* **Causal/Masked Attention**:

In [94]:
import torch

# Let's assume we have a sequence of length 4 (matching our earlier max_length)
seq_len = 4

# 1. Create a lower triangular mask of ones
# torch.tril extracts the lower triangular part of a matrix, zeroing out elements above the main diagonal
mask = torch.tril(torch.ones(seq_len, seq_len))
print("1. Lower Triangular Mask Matrix:")
print(mask)

# 2. To apply this to attention scores, we want to keep the values where mask is 1,
# and replace the 0s with -infinity so that softmax will turn them into 0 attention weights.
masked_scores = torch.ones(seq_len, seq_len) * 0.5  # Dummy raw attention scores

# Apply the mask: wherever the mask is 0, replace the score with -inf
masked_scores = masked_scores.masked_fill(mask == 0, float('-inf'))
print("\n2. Masked Attention Scores (before softmax):")
print(masked_scores)

# 3. Apply softmax to see how the attention weights are restricted to past tokens
attention_weights = torch.softmax(masked_scores, dim=-1)
print("\n3. Final Causal Attention Weights (after softmax):")
print(attention_weights)

1. Lower Triangular Mask Matrix:
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

2. Masked Attention Scores (before softmax):
tensor([[0.5000,   -inf,   -inf,   -inf],
        [0.5000, 0.5000,   -inf,   -inf],
        [0.5000, 0.5000, 0.5000,   -inf],
        [0.5000, 0.5000, 0.5000, 0.5000]])

3. Final Causal Attention Weights (after softmax):
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])


In [95]:
import math
import torch
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, max_len, dropout=0.0):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.dropout = nn.Dropout(dropout)

        # Register causal mask buffer so it's moved to GPU automatically with the model
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(max_len, max_len))
        )

    def forward(self, x):
        batch_size, seq_len, d_in = x.shape

        # Project inputs to Queries, Keys, and Values
        queries = self.W_query(x)  # (B, T, d_out)
        keys = self.W_key(x)      # (B, T, d_out)
        values = self.W_value(x)  # (B, T, d_out)

        # Compute scaled dot-product attention scores
        # (B, T, d_out) x (B, d_out, T) -> (B, T, T)
        attn_scores = torch.bmm(queries, keys.transpose(1, 2))
        attn_scores = attn_scores / math.sqrt(self.d_out)

        # Apply causal mask
        # Use self.mask[:seq_len, :seq_len] to accommodate variable sequence lengths up to max_len
        attn_scores = attn_scores.masked_fill(
            self.mask[:seq_len, :seq_len] == 0,
            float('-inf')
        )

        # Softmax to get attention weights
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Compute contextualized representations: (B, T, T) x (B, T, d_out) -> (B, T, d_out)
        context_vectors = torch.bmm(attn_weights, values)
        return context_vectors

# Quick test using the embeddings from the previous sections
causal_attention_layer = CausalAttention(d_in=256, d_out=16, max_len=4)
context_vectors = causal_attention_layer(final_embeddings_x)

print("Input final embeddings shape:  ", final_embeddings_x.shape)
print("Output causal context shape:   ", context_vectors.shape)

Input final embeddings shape:   torch.Size([4, 4, 256])
Output causal context shape:    torch.Size([4, 4, 16])


# Stage 2.4) **Multihead attention mechanism**


In [96]:
import math
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, max_len, num_heads, dropout=0.0, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        # Linear layers for projecting input to Query, Key, and Value
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Out projection layer to combine head outputs
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        # Register causal mask as a buffer
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(max_len, max_len))
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # Project inputs to Queries, Keys, and Values
        # Shape: (b, num_tokens, d_out)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Transform tensor shapes to split into multi-heads
        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        # Compute scaled dot-product attention scores
        # keys.transpose(2, 3) has shape (b, num_heads, head_dim, num_tokens)
        # attn_scores has shape (b, num_heads, num_tokens, num_tokens)
        attn_scores = torch.matmul(queries, keys.transpose(2, 3)) / math.sqrt(self.head_dim)

        # Apply the causal mask to prevent attending to future tokens
        attn_scores = attn_scores.masked_fill(
            self.mask[:num_tokens, :num_tokens] == 0,
            float('-inf')
        )

        # Softmax and Dropout
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Compute contextualized representation
        # (b, num_heads, num_tokens, num_tokens) x (b, num_heads, num_tokens, head_dim) -> (b, num_heads, num_tokens, head_dim)
        context_vec = torch.matmul(attn_weights, values)

        # Permute and concatenate heads back together
        # (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim) -> (b, num_tokens, d_out)
        context_vec = context_vec.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)

        # Final linear projection output
        return self.out_proj(context_vec)

# --- Demo of Multi-Head Attention ---
mha_layer = MultiHeadAttention(d_in=256, d_out=16, max_len=4, num_heads=2)
output_mha = mha_layer(final_embeddings_x)

print("Input embeddings shape: ", final_embeddings_x.shape)
print("Output MHA shape:        ", output_mha.shape)

Input embeddings shape:  torch.Size([4, 4, 256])
Output MHA shape:         torch.Size([4, 4, 16])


### Inspecting the Final Multi-Head Attention Outputs
Now, let's look at the actual output values generated by our multi-head attention layer `output_mha` corresponding to our initial pipeline embeddings.

In [97]:
# Print the detailed tensor values of our Multi-Head Attention output
print("=== Multi-Head Attention Output Tensor ===")
print("Shape:", output_mha.shape)
print("\nOutput Tensor (Batch 0):")
display(output_mha[0])

# Let's also check if there are any NaN or Inf values in the output
print("\nContains NaN:", torch.isnan(output_mha).any().item())
print("Contains Inf:", torch.isinf(output_mha).any().item())

=== Multi-Head Attention Output Tensor ===
Shape: torch.Size([4, 4, 16])

Output Tensor (Batch 0):


tensor([[-0.1485,  0.3127,  0.3460, -0.1125,  0.0301,  0.0703,  0.1029, -0.6245,
          1.1257,  0.9956, -0.0271,  0.7599, -0.2177, -0.3043, -0.1588, -0.3686],
        [ 0.2028, -0.1687,  0.4747,  0.2375, -0.0912,  0.1104,  0.2065, -0.5985,
          0.5911,  0.6016,  0.1498,  0.2196,  0.1110, -0.2656,  0.1199, -0.1190],
        [ 0.0799, -0.1800,  0.4713,  0.3001,  0.0436,  0.0361,  0.1471, -0.4150,
          0.6253,  0.4945,  0.1508,  0.1979,  0.3080, -0.2896,  0.2253, -0.2965],
        [ 0.1240, -0.0653,  0.5434,  0.3191,  0.0359,  0.1738,  0.2170, -0.4285,
          0.6893,  0.3940,  0.1461,  0.1635,  0.2306, -0.1792,  0.2936, -0.6126]],
       grad_fn=<SelectBackward0>)


Contains NaN: False
Contains Inf: False


### Decoding Multi-Head Attention Outputs Back to Text
To convert these dense output numerical values back into a readable text version, we can project the outputs back to our vocabulary space using a linear mapping (similar to a language model head) or compute the nearest matching embeddings.

In [98]:
# We will project the multi-head attention output back to the vocabulary dimension
# using our token embedding weights transpose (weight tying concept) to find the closest matching tokens.

# Project output (B, T, d_out) back to vocab size using a projection layer or matrix multiplication
# For demonstration, we map the 16-dimensional representation to our vocabulary logits
projection_layer = nn.Linear(16, vocab_size)

with torch.no_grad():
    logits = projection_layer(output_mha)
    predicted_token_ids = torch.argmax(logits, dim=-1)

print("=== Predicted Token IDs from MHA Output ===")
print(predicted_token_ids)

print("\n=== Decoded Text Version ===")
for i in range(batch_size):
    decoded_str = tokenizer.decode(predicted_token_ids[i].tolist())
    print(f"Sequence {i+1} Text: {repr(decoded_str)}")

=== Predicted Token IDs from MHA Output ===
tensor([[50017, 31713, 31713, 31713],
        [38825, 31713, 31713, 31713],
        [31713, 35382, 36653, 38825],
        [46753, 26831, 26831, 15528]])

=== Decoded Text Version ===
Sequence 1 Text: ' flared declarations declarations declarations'
Sequence 2 Text: 'Solar declarations declarations declarations'
Sequence 3 Text: ' declarations wartimechangesSolar'
Sequence 4 Text: ' pasture redemption redemption Tampa'


### Stage 3 : LLM Architecture

### Theoretical Representation of a Transformer Block

A **Transformer Block** (often referred to as a **GPT Block** or **Decoder Block**) is the repeating unit of modern Large Language Models. It processes contextual representations by combining token relations (via multi-head self-attention) and token features (via feed-forward networks).

Here is the mathematical formulation of how an input vector $X$ transitions through a single Transformer block:

---

#### 1. The Block Equation (GPT-style Pre-LayerNorm)
Most modern decoders use **Pre-LayerNorm**, where normalization is applied before the sub-layers rather than after:

$$X_{1} = X + \text{MultiHeadAttention}(\text{LayerNorm}(X))$$
$$X_{2} = X_{1} + \text{FeedForward}(\text{LayerNorm}(X_{1}))$$

Where:
* $X$ is the incoming tensor representation of shape `(Batch, Seq_Len, d_model)`.
* $X_{1}$ is the intermediate representation after attention and residual adding.
* $X_{2}$ is the final output of the block, ready to go to the next block.

---

#### 2. Key Components Breakdown

##### A. Pre-Layer Normalization (`LayerNorm`)
Normalizes activations across the last dimension (the feature dimension $D$) to maintain stable gradient flows:
$$\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta$$
Where $\mu$ and $\sigma^2$ are the mean and variance of the vector, $\epsilon$ is a tiny constant for numerical stability, and $\gamma, \beta$ are learnable scale and shift parameters.

##### B. Multi-Head Causal Attention (`MHA`)
Mixes semantic context across time (tokens) while preventing information leakage from the future using a causal lower-triangular mask:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}} + M\right)V$$
* Where $M$ is the causal mask matrix ($0$ for valid positions, $-\infty$ for future positions).

##### C. Feed-Forward Network (`FFN`)
Processes each token position individually using fully connected projection layers with non-linear activation:
$$\text{FFN}(x) = \text{Linear}_2(\text{GELU}(\text{Linear}_1(x)))$$
* Typically, $\text{Linear}_1$ projects the hidden state to a higher dimension (often $4 \times d_{\text{model}}$), and $\text{Linear}_2$ projects it back to $d_{\text{model}}$.

##### D. Residual Connections (Skip Connections)
Notice the addition $+ X$ and $+ X_{1}$ in the block equations. These direct connections allow gradients to flow backwards through stacked blocks without vanishing, making it possible to train networks with hundreds of layers.

### 3. Structural Flow Diagram of a Transformer Block

Here is how a tensor flows through a modern **Pre-LayerNorm Transformer Block** (GPT-style):

```text
                        Input: X
                (Batch, Seq_Len, d_model)
               /                         \
              / (Residual Connection)     \
             |                             v
             |                        LayerNorm(X)
             |                             |
             |                             v
             |                    Multi-Head Attention (MHA)
             |                             |
             |                             v
             |                       Dropout / Projection
             |                             |
             \                             /
              \---->  [+] <---------------/
                       |
                   X_intermediate (X_1)
                  /                    \
                 / (Residual Connection)\
                |                        v
                |                   LayerNorm(X_1)
                |                        |
                |                        v
                |                Feed-Forward Net (FFN)
                |                 - Linear Projection Up (4 * d_model)
                |                 - Activation (GELU)
                |                 - Linear Projection Down (d_model)
                |                        |
                \                        /
                 \--->  [+] <-----------/
                         |
                      Output: X_2
               (Batch, Seq_Len, d_model)
```

### Key Pipeline Phases:
1. **Pre-LN Attention Branch:** The normalized input goes into the MHA block. Only contextual relations across tokens are computed here. The output is added back to the original input vector $X$ via a skip connection to produce the intermediate tensor $X_1$.
2. **Pre-LN Feed-Forward Branch:** The normalized intermediate vector $X_1$ goes into the FFN. Individual token representations are projected up to a higher-dimensional space to combine features, activated, and projected back. This output is added back to $X_1$ via another skip connection to yield $X_2$.

### GPT-2 Model Configurations

When scaling up to standard GPT-2 architectures, we use a specific set of hyperparameters. Here is the configuration breakdown for the different pre-trained scales of GPT-2:

| Model Parameter | GPT-2 Small (124M) | GPT-2 Medium (355M) | GPT-2 Large (774M) | GPT-2 XL (1.5B) |
| :--- | :--- | :--- | :--- | :--- |
| **Embedding Dim ($d_{\text{model}}$)** | 768 | 1024 | 1280 | 1600 |
| **Number of Layers ($n_{\text{layer}}$)** | 12 | 24 | 36 | 48 |
| **Number of Heads ($n_{\text{head}}$)** | 12 | 16 | 20 | 25 |
| **Context Window ($n_{\text{ctx}}$)** | 1024 | 1024 | 1024 | 1024 |
| **Vocab Size ($V$)** | 50,257 | 50,257 | 50,257 | 50,257 |

Let's define a PyTorch-friendly configuration dictionary and class that we can use to build our eventual model structure.

In [101]:
# Defining GPT-2 configuration parameters as a dictionary and a class config

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Maximum context sequence length
    "emb_dim": 768,         # Embedding dimension (d_model)
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of Transformer blocks
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias setting
}

# Alternative Object-Oriented Config Class
class GPTConfig:
    def __init__(self, vocab_size=50257, context_length=1024, emb_dim=768, n_heads=12, n_layers=12, drop_rate=0.1, qkv_bias=False):
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.emb_dim = emb_dim
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.drop_rate = drop_rate
        self.qkv_bias = qkv_bias

# Instantiate standard GPT-2 configuration
cfg_124m = GPTConfig(**GPT_CONFIG_124M)
print(f"GPT-2 configuration initialised successfully!")
print(f"- Vocab Size: {cfg_124m.vocab_size}")
print(f"- Embedding Dimension: {cfg_124m.emb_dim}")
print(f"- Layers: {cfg_124m.n_layers} blocks")
print(f"- Attention Heads: {cfg_124m.n_heads}")

GPT-2 configuration initialised successfully!
- Vocab Size: 50257
- Embedding Dimension: 768
- Layers: 12 blocks
- Attention Heads: 12


### 4. Layer Normalization & Feed-Forward Network Blocks

To build our complete GPT-style Transformer block, we will implement the two remaining core modules:
1. **Layer Normalization (`LayerNorm`)** with scale (gamma) and shift (beta) parameters.
2. **Feed-Forward Network (`FeedForward`)** using standard linear layers and the GELU activation function.

In [104]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        # Learnable scale (gamma) and shift (beta) parameters
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        # x shape: (batch_size, seq_len, emb_dim)
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * normalized + self.shift

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Linear expansion up to 4 * emb_dim, and projection back to emb_dim
        self.layers = nn.Sequential(
            nn.Linear(cfg.emb_dim, 4 * cfg.emb_dim),
            nn.GELU(),
            nn.Linear(4 * cfg.emb_dim, cfg.emb_dim),
            nn.Dropout(cfg.drop_rate)
        )

    def forward(self, x):
        return self.layers(x)

# Quick validation using our configuration
ln_test = LayerNorm(emb_dim=cfg_124m.emb_dim)
ff_test = FeedForward(cfg_124m)

# Let's verify with random inputs of matching shape: (batch_size=2, seq_len=4, emb_dim=768)
dummy_input = torch.randn(2, 4, cfg_124m.emb_dim)
output_ln = ln_test(dummy_input)
output_ff = ff_test(output_ln)

print("Input tensor shape:      ", dummy_input.shape)
print("LayerNorm output shape:  ", output_ln.shape)
print("Feed-Forward output shape:", output_ff.shape)

Input tensor shape:       torch.Size([2, 4, 768])
LayerNorm output shape:   torch.Size([2, 4, 768])
Feed-Forward output shape: torch.Size([2, 4, 768])


### 5. Building the Complete Transformer Block

We will now assemble our `TransformerBlock` module. It combines:
- **`LayerNorm`** (Pre-LayerNorm pattern)
- **`MultiHeadAttention`** (Causal multi-head self-attention)
- **`FeedForward`** (Position-wise MLP block)
- **Residual Connections** ($+ x$ shortcut connections)

In [105]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Attention sub-layer components
        self.ln1 = LayerNorm(cfg.emb_dim)
        self.att = MultiHeadAttention(
            d_in=cfg.emb_dim,
            d_out=cfg.emb_dim,
            max_len=cfg.context_length,
            num_heads=cfg.n_heads,
            dropout=cfg.drop_rate,
            qkv_bias=cfg.qkv_bias
        )

        # Feed-Forward sub-layer components
        self.ln2 = LayerNorm(cfg.emb_dim)
        self.ff = FeedForward(cfg)

    def forward(self, x):
        # 1. Attention Block with Pre-LayerNorm and Residual Connection
        x = x + self.att(self.ln1(x))

        # 2. Feed-Forward Block with Pre-LayerNorm and Residual Connection
        x = x + self.ff(self.ln2(x))

        return x

# Quick verification of the completed Transformer Block
tb_layer = TransformerBlock(cfg_124m)

# Input size matching standard GPT-2 configuration: (batch_size=2, seq_len=4, emb_dim=768)
dummy_embeddings = torch.randn(2, 4, cfg_124m.emb_dim)
tb_output = tb_layer(dummy_embeddings)

print("Input block shape: ", dummy_embeddings.shape)
print("Output block shape:", tb_output.shape)

Input block shape:  torch.Size([2, 4, 768])
Output block shape: torch.Size([2, 4, 768])


### 6. The Complete GPT Model Architecture

Below is the complete consolidated code combining all implemented components into the final **GPTModel** class. This includes our custom `LayerNorm`, `FeedForward`, `MultiHeadAttention`, `TransformerBlock`, and the ultimate model wrapper containing both token and positional embedding layers and a Language Model (LM) Head.

In [106]:
import math
import torch
import torch.nn as nn

# --- 1. Model Configurations ---
class GPTConfig:
    def __init__(self, vocab_size=50257, context_length=1024, emb_dim=768, n_heads=12, n_layers=12, drop_rate=0.1, qkv_bias=False):
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.emb_dim = emb_dim
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.drop_rate = drop_rate
        self.qkv_bias = qkv_bias

# --- 2. Custom Layer Normalization ---
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * normalized + self.shift

# --- 3. Custom Feed-Forward Network Block ---
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg.emb_dim, 4 * cfg.emb_dim),
            nn.GELU(),
            nn.Linear(4 * cfg.emb_dim, cfg.emb_dim),
            nn.Dropout(cfg.drop_rate)
        )

    def forward(self, x):
        return self.layers(x)

# --- 4. Custom Multi-Head Causal Attention ---
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, max_len, num_heads, dropout=0.0, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            'mask',
            torch.tril(torch.ones(max_len, max_len))
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = torch.matmul(queries, keys.transpose(2, 3)) / math.sqrt(self.head_dim)
        attn_scores = attn_scores.masked_fill(
            self.mask[:num_tokens, :num_tokens] == 0,
            float('-inf')
        )

        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = torch.matmul(attn_weights, values)
        context_vec = context_vec.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

# --- 5. Unified Transformer Block ---
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = LayerNorm(cfg.emb_dim)
        self.att = MultiHeadAttention(
            d_in=cfg.emb_dim,
            d_out=cfg.emb_dim,
            max_len=cfg.context_length,
            num_heads=cfg.n_heads,
            dropout=cfg.drop_rate,
            qkv_bias=cfg.qkv_bias
        )
        self.ln2 = LayerNorm(cfg.emb_dim)
        self.ff = FeedForward(cfg)

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# --- 6. Full GPT Model Class ---
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.emb_dim)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.emb_dim)
        self.drop_emb = nn.Dropout(cfg.drop_rate)

        # Sequential stack of multiple Transformer blocks
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg.n_layers)]
        )

        self.final_ln = LayerNorm(cfg.emb_dim)
        self.out_head = nn.Linear(cfg.emb_dim, cfg.vocab_size, bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(tok_embeds + pos_embeds)

        x = self.trf_blocks(x)
        x = self.final_ln(x)

        logits = self.out_head(x)
        return logits

# Verification of assembly
cfg_124m = GPTConfig()
gpt_model = GPTModel(cfg_124m)

# Run dummy pass to test full integration
dummy_input = torch.randint(0, cfg_124m.vocab_size, (2, 4))
out_logits = gpt_model(dummy_input)
print("Unified GPT Model successfully integrated and tested!")
print("Input batch shape:", dummy_input.shape)
print("Output logits shape:", out_logits.shape)

Unified GPT Model successfully integrated and tested!
Input batch shape: torch.Size([2, 4])
Output logits shape: torch.Size([2, 4, 50257])


### Stage 4: Next-Token Generation Loop

To make our `GPTModel` generate readable text, we must implement a **generation loop**.

During text generation:
1. We take a starting prompt string and encode it into token IDs.
2. We feed these IDs into our model to obtain the output `logits` for the last token in the sequence.
3. We apply a decoding strategy (like **Greedy Decoding**, where we pick the token index with the highest probability score using `torch.argmax`) to determine the next token.
4. We append the predicted token ID to our sequence and repeat the process until we reach our target length or generate an end-of-text token.

In [107]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is a batch of input token indices of shape (batch, seq_len)
    for _ in range(max_new_tokens):
        # Crop current context if it exceeds the maximum supported sequence length
        idx_cond = idx[:, -context_size:]

        # Get predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last token's logits: shape becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Greedy decoding: select token index with the highest logit value
        next_idx = torch.argmax(logits, dim=-1, keepdim=True)

        # Append the predicted index to our running sequence
        idx = torch.cat((idx, next_idx), dim=1)

    return idx

# Let's test the generation loop on our untrained model!
# 1. Initialize our tokenizer using the GPT-2 tiktoken encoder
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

# 2. Define a prompt and encode it
start_context = "The future of artificial intelligence is"
encoded_prompt = torch.tensor([tokenizer.encode(start_context)])

# 3. Generate 10 new tokens
gpt_model.eval() # Set model to evaluation mode (disables dropout)
generated_ids = generate_text_simple(
    model=gpt_model,
    idx=encoded_prompt,
    max_new_tokens=10,
    context_size=cfg_124m.context_length
)

# 4. Decode the output IDs back to text
generated_text = tokenizer.decode(generated_ids[0].tolist())
print("Prompt text:    ", start_context)
print("Generated text:", generated_text)
print("\n(Note: The generated text looks like random gibberish because the model weights are currently randomized and untrained.)")

Prompt text:     The future of artificial intelligence is
Generated text: The future of artificial intelligence is pont Experiment billergicoles Hear arrival obliged Specific diver

(Note: The generated text looks like random gibberish because the model weights are currently randomized and untrained.)


### Stage 5: Training Loop on 'The Verdict'

Now we will implement the pre-training loop for our GPT model using the preprocessed text from Edith Wharton's *The Verdict*.

To do this, we will:
1. **Prepare the Data:** Instantiate a `TextDataset` and `DataLoader` over the complete text (`raw_text`) using the standard `tiktoken` BPE tokenizer.
2. **Define Loss & Optimization:** Use standard **Cross-Entropy Loss** to evaluate the model's next-token predictions, and the **AdamW** optimizer for weight updates.
3. **Implement the Training Loop:** Write a structured function to calculate loss, perform backpropagation, update parameters, and output progress statistics over several epochs.

In [108]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# 1. Re-create the DataLoader over the entire text using BPE tokenizer
# We configure a context length of 256 for a faster training run on Colab
custom_config = GPTConfig(
    vocab_size=50257,
    context_length=256,
    emb_dim=768,
    n_heads=12,
    n_layers=12,
    drop_rate=0.1,
    qkv_bias=False
)

train_dataset = TextDataset(
    txt=raw_text,
    tokenizer=tokenizer,
    max_length=custom_config.context_length,
    stride=128 # 50% overlap
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    drop_last=True
)

# 2. Instantiate a fresh model with our custom training configuration
model = GPTModel(custom_config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3. Define Optimizer and Loss function
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
loss_fn = nn.CrossEntropyLoss()

# 4. Training Loop Function
def train_model_simple(model, train_loader, optimizer, loss_fn, device, epochs=2):
    model.train()
    print(f"Training initiated on device: {device}\n")

    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            # Forward pass: get predicted logits shape (batch, seq_len, vocab_size)
            logits = model(x)

            # Cross-Entropy expects flattened logits (batch * seq_len, vocab_size)
            # and target indices of shape (batch * seq_len)
            loss = loss_fn(logits.view(-1, logits.size(-1)), y.view(-1))

            # Backward pass & optimization
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1:02d} | Avg Loss: {avg_loss:.4f}")

# Train the model for 5 epochs
train_model_simple(model, train_loader, optimizer, loss_fn, device, epochs=5)

Training initiated on device: cpu

Epoch 01 | Avg Loss: 7.6961
Epoch 02 | Avg Loss: 5.8670
Epoch 03 | Avg Loss: 4.9741
Epoch 04 | Avg Loss: 3.7175
Epoch 05 | Avg Loss: 2.4265


### Stage 6: Testing our Trained Model

Now that our model weights are no longer purely random, let's run our custom next-token generation loop on our trained model using the same starting prompt to observe the progress.

In [112]:
# 1. Set the trained model to evaluation mode
model.eval()

# 2. Encode our test prompt
test_prompt = "Because words are split into subwords, the length of your dataset"
encoded_test = torch.tensor([tokenizer.encode(test_prompt)]).to(device)

# 3. Generate 15 new tokens using our trained weights
generated_ids = generate_text_simple(
    model=model,
    idx=encoded_test,
    max_new_tokens=15,
    context_size=custom_config.context_length
)

# 4. Decode and print the output
typed_output = tokenizer.decode(generated_ids[0].tolist())
print("Prompt text:    ", test_prompt)
print("Generated text:", typed_output)

Prompt text:     Because words are split into subwords, the length of your dataset
Generated text: Because words are split into subwords, the length of your dataset-dancers" to see how he had been, with your
"


### Model Evaluation: Train/Val Loss, Perplexity, and Fluency Metrics
We will evaluate our trained model by splitting our text data into training (90%) and validation (10%) sets, tracking perplexity, and generating comparative sample predictions.

In [113]:
import math
import torch

# 1. Create a validation split (last 10% of raw_text characters)
split_idx = int(len(raw_text) * 0.9)
train_text = raw_text[:split_idx]
val_text = raw_text[split_idx:]

# 2. Instantiate datasets
eval_train_dataset = TextDataset(
    txt=train_text,
    tokenizer=tokenizer,
    max_length=custom_config.context_length,
    stride=128
)
eval_val_dataset = TextDataset(
    txt=val_text,
    tokenizer=tokenizer,
    max_length=custom_config.context_length,
    stride=128
)

eval_train_loader = DataLoader(eval_train_dataset, batch_size=2, shuffle=False)
eval_val_loader = DataLoader(eval_val_dataset, batch_size=2, shuffle=False)

# 3. Evaluation Function
def evaluate_model(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits.view(-1, logits.size(-1)), y.view(-1))
            total_loss += loss.item()

    avg_loss = total_loss / max(1, len(data_loader))
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity

# Calculate metrics
train_loss, train_ppl = evaluate_model(model, eval_train_loader, loss_fn, device)
val_loss, val_ppl = evaluate_model(model, eval_val_loader, loss_fn, device)

print("=== GPT Model Evaluation Metrics ===")
print(f"Training Loss:   {train_loss:.4f} | Perplexity: {train_ppl:.2f}")
print(f"Validation Loss: {val_loss:.4f} | Perplexity: {val_ppl:.2f}")


=== GPT Model Evaluation Metrics ===
Training Loss:   1.4833 | Perplexity: 4.41
Validation Loss: 3.9509 | Perplexity: 51.98
